In [1]:
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from rich import print, pretty, inspect
from rich.console import Console

In [2]:
console = Console()
console.print("hello", style = "bold white")

hello

### 0. Extract All Lines Modifying ```this_block```
Example wrapper file used for dev: [casper_wb_fft_config.m](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m)

Wei Liu's example SciLab port for ```casper_wb_fft_config.m```:
1. JSON: [wbfft.json](https://github.com/liuweiseu/mlib_devel/blob/c74c7bae6b34bd3d8231a78a75955ab2a48cf24e/scilab_library/scilab_blocks/casper_dsp/wbfft.json)
2. SciLab: [wbfft.sci](https://github.com/liuweiseu/mlib_devel/blob/c74c7bae6b34bd3d8231a78a75955ab2a48cf24e/scilab_library/scilab_blocks/casper_dsp/wbfft.sci)
3. Python: [wbfft.py](https://github.com/liuweiseu/mlib_devel/blob/c74c7bae6b34bd3d8231a78a75955ab2a48cf24e/scilab_library/dsp_blocks/wbfft.py)

In [3]:
wrappers_root = Path(".")
scilab_dsp_root = "casper_dspdevel"

def match_target_lines(target_fpath, line_pat):
    with open(target_fpath, "r") as f:
        d = f.read()
        matches = re.findall(line_pat, d)
        match_line_df = pd.DataFrame(matches)
    return match_line_df

def match_this_block(target_file, this_block_pat="this_block\.(.*)\((.*)\);"):
    this_block_lines_df = match_target_lines(
        target_file,
        line_pat=this_block_pat
    )
    return this_block_lines_df.rename(columns={0: "fn", 1: "args"})

In [4]:
target_file = "casper_wb_fft_config.m"
target_fpath = wrappers_root / target_file

### 1. Extract File dependencies

Regex patterns ([wbfft example](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m#L246))

1. Find [import lines](https://regex101.com/r/gilGkt/1): ```this_block.addFileToLibrary(...);```
2. Match and extract file paths: 

In [5]:
## Extract file dependencies


# Regular expressions
dep_fpaths_pat = "\[filepath '(.*)'\]"
relative_path_pat = "[/\.]*(.*)"

# Load relevant lines
df = match_this_block(target_fpath)

# Filter by function
dep_fns = ["addFileToLibrary"]
dep_df = df[df['fn'].isin(dep_fns)]

# Extract arguments + rename
dep_args_df = dep_df['args'].str.split(',', expand=True)
dep_args_df = dep_args_df.rename(columns={0: "simulink_fpath", 1: "lib"})
dep_df = pd.concat([dep_args_df, dep_df], axis=1)

# Extract simulink fpaths and translate to scilab fpaths
dep_df['simulink_fpath'] = dep_df['simulink_fpath'].str.extract(dep_fpaths_pat)
dep_df['scilab_fpath'] = scilab_dsp_root + "/" + (dep_df['simulink_fpath'].str.extract(relative_path_pat))

# Drop any imports that don't have first argument starting with "filename"
dep_df = dep_df.dropna().reset_index(drop=True)

dep_df.head(5)

,simulink_fpath,lib,fn,args,scilab_fpath
0,/../../common_pkg/fixed_float_types_c.vhd,'common_pkg_lib',addFileToLibrary,[filepath '/../../common_pkg/fixed_float_types...,casper_dspdevel/common_pkg/fixed_float_types_c...
1,/../../common_pkg/fixed_pkg_c.vhd,'common_pkg_lib',addFileToLibrary,[filepath '/../../common_pkg/fixed_pkg_c.vhd']...,casper_dspdevel/common_pkg/fixed_pkg_c.vhd
2,/../../common_pkg/common_pkg.vhd,'common_pkg_lib',addFileToLibrary,"[filepath '/../../common_pkg/common_pkg.vhd'],...",casper_dspdevel/common_pkg/common_pkg.vhd
3,/../../common_components/common_pipeline.vhd,'common_components_lib',addFileToLibrary,[filepath '/../../common_components/common_pip...,casper_dspdevel/common_components/common_pipel...
4,/../../casper_adder/common_add_sub.vhd,'casper_adder_lib',addFileToLibrary,[filepath '/../../casper_adder/common_add_sub....,casper_dspdevel/casper_adder/common_add_sub.vhd


### 2. Extract Generic Parameters
[Comment on supported generics](https://github.com/talonmyburgh/casper_dspdevel/blob/a8b6f3d9311d7f9579a6eb7015d4d549fece36fe/wrappers/simulink/casper_wb_fft_config.m#L222):
> %      The addGeneric function takes  3 parameters, generic name, type and constant value.
> Supported types are boolean, real, integer and string.

Regex patterns

1. Find lines with generics.
2. Match and extract file paths: 

In [23]:
## Extract generics

# Load "subsystem mask parameters for dynamic ports" (i.e. parent-controlled generics)
parent_params_df = match_target_lines(
    target_fpath,
    line_pat = "get_param\((.*),(.*)\);"
)
parent_params_df = parent_params_df.rename(columns={0: 'block_name', 1: 'parent_generic_name'})

# Load this_block params
this_block_params_df = match_target_lines(
    target_fpath,
    line_pat = "addGeneric\((.*),(.*),(.*)\);"
)

this_block_params_df = this_block_params_df.rename(columns={0: 'generic_name', 1: 'type', 2: 'constant_value'})

# Combine parameters into one DataFrame
gen_df = pd.concat([this_block_params_df, parent_params_df])

gen_df

,generic_name,type,constant_value,block_name,parent_generic_name
0,'use_reorder','boolean',bool2str(use_reorder),NaN,NaN
1,'use_fft_shift','boolean',bool2str(use_fft_shift),NaN,NaN
2,'use_separate','boolean',bool2str(use_separate),NaN,NaN
3,'alt_output','boolean',bool2str(alt_output),NaN,NaN
4,'wb_factor','natural',wb_factor,NaN,NaN
5,'nof_points','natural',nof_points,NaN,NaN
6,'in_dat_w','natural',i_d_w,NaN,NaN
7,'out_dat_w','natural',o_d_w,NaN,NaN
8,'out_gain_w','natural',o_g_w,NaN,NaN
9,'stage_dat_w','natural',s_d_w,NaN,NaN


### Visualize Python File AST

In [29]:
# In Jupyter notebook
from IPython.display import Image
import ast
from graphviz import Digraph

def extract_ast(code):
    """Extract abstract syntax tree from Python code"""
    tree = ast.parse(code)
    return tree

def visualize_ast(tree):
    """Create Graphviz visualization of AST"""
    dot = Digraph(comment='AST', node_attr={'shape': 'box', 'style': 'filled', 'fillcolor': '#f0f0f0'})
    
    def add_nodes_edges(node, parent=None):
        node_name = str(id(node))
        label = type(node).__name__
        
        # Add special handling for common node types
        if isinstance(node, ast.ClassDef):
            dot.node(node_name, f"Class: {node.name}", fillcolor='#e0f0e0')
        elif isinstance(node, ast.FunctionDef):
            dot.node(node_name, f"Function: {node.name}", fillcolor='#e0e0f0')
        elif isinstance(node, ast.Name):
            dot.node(node_name, f"Name: {node.id}", fillcolor='#f0e0f0')
        else:
            dot.node(node_name, label)
            
        if parent:
            dot.edge(parent, node_name)
            
        for child in ast.iter_child_nodes(node):
            add_nodes_edges(child, node_name)
            
    add_nodes_edges(tree)
    return dot

# Usage example
"""
with open('wbfft.py', 'r') as f:
    code = f.read()

# 1. Extract AST
ast_tree = extract_ast(code)
print(ast.dump(ast_tree, indent=4))  # Text representation

# 2. Visualize AST
dot = visualize_ast(ast_tree)
dot.render('ast_graph', format='png', cleanup=True)  # Save as PNG
dot  # Display in notebook if using Jupyter

Image(dot.render(format='png'))
"""


"\nwith open('wbfft.py', 'r') as f:\n    code = f.read()\n\n# 1. Extract AST\nast_tree = extract_ast(code)\nprint(ast.dump(ast_tree, indent=4))  # Text representation\n\n# 2. Visualize AST\ndot = visualize_ast(ast_tree)\ndot.render('ast_graph', format='png', cleanup=True)  # Save as PNG\ndot  # Display in notebook if using Jupyter\n\nImage(dot.render(format='png'))\n"